In [18]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
import os
import json
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

import warnings
warnings.filterwarnings('ignore')


In [20]:
# (CHANGE THESE ACCORDING TO YOUR DRIVE)
STUDENTLIFE_PATH = "/content/drive/MyDrive/AAI/Data"
#OUTPUT_PATH = "/content/drive/MyDrive/AI_Burnout_Predictor/results_realistic_studentlife"

#os.makedirs(OUTPUT_PATH, exist_ok=True)

print("Paths configured.")


Paths configured.


In [21]:
# Loading StudentLife data
def load_studentlife_json(folder_path):
    all_data = []
    for file_name in os.listdir(folder_path):
        if file_name.endswith(".json"):
            student_id = file_name.replace(".json", "")
            with open(os.path.join(folder_path, file_name), "r") as f:
                records = json.load(f)
                for r in records:
                    r["student_id"] = student_id
                    all_data.append(r)
    return pd.DataFrame(all_data)

print("Loading StudentLife...")

stress_raw = load_studentlife_json(os.path.join(STUDENTLIFE_PATH, "Stress"))
activity_raw = load_studentlife_json(os.path.join(STUDENTLIFE_PATH, "Activity"))
sleep_raw = load_studentlife_json(os.path.join(STUDENTLIFE_PATH, "Sleep"))

print(f"Stress: {stress_raw.shape}")
print(f"Activity: {activity_raw.shape}")
print(f"Sleep: {sleep_raw.shape}")


Loading StudentLife...
Stress: (2408, 5)
Activity: (833, 9)
Sleep: (1644, 7)


In [22]:
stress_raw.head()

,level,location,resp_time,student_id,null
0,2,"43.70644884,-72.28820168",1364609268,Stress_u10,NaN
1,1,"43.70241112,-72.28788985",1364800742,Stress_u10,NaN
2,1,"43.70243257,-72.28772434",1364770637,Stress_u10,NaN
3,NaN,NaN,1364121099,Stress_u10,4
4,NaN,NaN,1364118564,Stress_u10,"43.70621142,-72.28697323"


In [23]:
activity_raw.head()

,Social2,null,resp_time,student_id,other_relaxing,other_working,relaxing,working,location
0,2,1,1365015406,Activity_u30,NaN,NaN,NaN,NaN,NaN
1,2,2,1364584997,Activity_u30,NaN,NaN,NaN,NaN,NaN
2,2,1,1364757470,Activity_u30,NaN,NaN,NaN,NaN,NaN
3,3,1,1365015405,Activity_u30,NaN,NaN,NaN,NaN,NaN
4,3,1,1365285715,Activity_u30,NaN,NaN,NaN,NaN,NaN


In [24]:
sleep_raw.head()

,null,resp_time,student_id,hour,location,rate,social
0,1,1364121437,Sleep_u54,NaN,NaN,NaN,NaN
1,1,1364118985,Sleep_u54,NaN,NaN,NaN,NaN
2,1,1364121435,Sleep_u54,NaN,NaN,NaN,NaN
3,1,1364121434,Sleep_u54,NaN,NaN,NaN,NaN
4,1,1364118978,Sleep_u54,NaN,NaN,NaN,NaN


In [25]:
# Cleaning the STRESS DATASET
# =========================
print("\n DATASET: STRESS")
print("Shows self-reported student stress levels over time")

print("\nCleaning StudentLife Stress...")

stress_clean = stress_raw.copy()

# Dropping the 'null' column
if 'null' in stress_clean.columns:
    stress_clean = stress_clean.drop(columns=['null'])

print("\nInitial Stress Dataset:")
print(stress_clean)

# Converting  timestamp
stress_clean['timestamp'] = pd.to_datetime(stress_clean['resp_time'], unit='s')
print("\nAfter converting resp_time to timestamp:")
print(stress_clean)

# Cleaning  student_id
stress_clean['student_id'] = stress_clean['student_id'].str.replace('Stress_', '')
print("\nAfter cleaning student_id:")
print(stress_clean)



 DATASET: STRESS
Shows self-reported student stress levels over time

Cleaning StudentLife Stress...

Initial Stress Dataset:
     level                  location   resp_time  student_id
0        2  43.70644884,-72.28820168  1364609268  Stress_u10
1        1  43.70241112,-72.28788985  1364800742  Stress_u10
2        1  43.70243257,-72.28772434  1364770637  Stress_u10
3      NaN                       NaN  1364121099  Stress_u10
4      NaN                       NaN  1364118564  Stress_u10
...    ...                       ...         ...         ...
2403     1  43.69230566,-72.26694933  1369244504  Stress_u44
2404     1  43.69213441,-72.26712004  1369329213  Stress_u44
2405     1  43.69222786,-72.26719913  1369416038  Stress_u44
2406     2   43.69341383,-72.2733575  1369675715  Stress_u44
2407     5  43.69232465,-72.26672118  1370034952  Stress_u44

[2408 rows x 4 columns]

After converting resp_time to timestamp:
     level                  location   resp_time  student_id  \
0        2

In [26]:
#renaming the level column
stress_clean = stress_clean.rename(columns={'level': 'stress_level'})
print("\nAfter renaming level → stress_level:")
print(stress_clean)

#Converting stress_level to numeric data
stress_clean['stress_level'] = pd.to_numeric(stress_clean['stress_level'], errors='coerce')
print("\nAfter converting stress_level to numeric:")
print(stress_clean)

if 'null' in stress_clean.columns:

    # if stress_level is missing but null looks like numeric, then use this
    null_as_num = pd.to_numeric(stress_clean['null'], errors='coerce')
    fill_mask = stress_clean['stress_level'].isna() & null_as_num.notna()
    if fill_mask.any():
        stress_clean.loc[fill_mask, 'stress_level'] = null_as_num.loc[fill_mask]

    #If location is missing but null looks like "lat,long", then use this
    if 'location' in stress_clean.columns:
        null_as_str = stress_clean['null'].astype(str)
        coord_mask = stress_clean['location'].isna() & null_as_str.str.match(
            r'^-?\d+(\.\d+)?,-?\d+(\.\d+)?$'
        )
        if coord_mask.any():
            stress_clean.loc[coord_mask, 'location'] = stress_clean.loc[coord_mask, 'null']

print("\nAfter recovering values from 'null' (if applicable):")
print(stress_clean)

#dropping missing stress values
stress_clean = stress_clean.dropna(subset=['stress_level'])
print("\nAfter dropping NaN stress levels:")
print(stress_clean)

#Explicit float conversion
stress_clean['stress_level'] = stress_clean['stress_level'].astype(float)
print("\nFinal cleaned Stress dataset:")
print(stress_clean)



After renaming level → stress_level:
     stress_level                  location   resp_time student_id  \
0               2  43.70644884,-72.28820168  1364609268        u10   
1               1  43.70241112,-72.28788985  1364800742        u10   
2               1  43.70243257,-72.28772434  1364770637        u10   
3             NaN                       NaN  1364121099        u10   
4             NaN                       NaN  1364118564        u10   
...           ...                       ...         ...        ...   
2403            1  43.69230566,-72.26694933  1369244504        u44   
2404            1  43.69213441,-72.26712004  1369329213        u44   
2405            1  43.69222786,-72.26719913  1369416038        u44   
2406            2   43.69341383,-72.2733575  1369675715        u44   
2407            5  43.69232465,-72.26672118  1370034952        u44   

               timestamp  
0    2013-03-30 02:07:48  
1    2013-04-01 07:19:02  
2    2013-03-31 22:57:17  
3    2013-03-

In [27]:
# Code for Activity data cleaning
print("\n DATASET: ACTIVITY")
print(" Shows students' daily activity levels (social, working, relaxing)")

print("\nCleaning StudentLife Activity")

activity_clean = activity_raw.copy()

# Drop the 'null' coloumns
if 'null' in activity_clean.columns:
    activity_clean = activity_clean.drop(columns=['null'])

print("\nInitial Activity Dataset:")
print(activity_clean)

# Convert timestamp

activity_clean['timestamp'] = pd.to_datetime(activity_clean['resp_time'], unit='s')
print("\nAfter converting resp_time to timestamp:")
print(activity_clean)

# Clean student_id
activity_clean['student_id'] = activity_clean['student_id'].str.replace('Activity_', '')
print("\nAfter cleaning student_id:")
print(activity_clean)



 DATASET: ACTIVITY
 Shows students' daily activity levels (social, working, relaxing)

Cleaning StudentLife Activity

Initial Activity Dataset:
    Social2   resp_time    student_id other_relaxing other_working relaxing  \
0         2  1365015406  Activity_u30            NaN           NaN      NaN   
1         2  1364584997  Activity_u30            NaN           NaN      NaN   
2         2  1364757470  Activity_u30            NaN           NaN      NaN   
3         3  1365015405  Activity_u30            NaN           NaN      NaN   
4         3  1365285715  Activity_u30            NaN           NaN      NaN   
..      ...         ...           ...            ...           ...      ...   
828     NaN  1368744712  Activity_u58              1             2        5   
829     NaN  1368237572  Activity_u58              1             1        5   
830     NaN  1368571581  Activity_u58              1             2        5   
831     NaN  1368828978  Activity_u58              3             

In [28]:
# Sleep dataset processing

# Print dataset name
print("\ DATASET: SLEEP")

# Describe dataset purpose
print("Shows students' self-reported sleep duration in hours")

# Start cleaning process
print("\nCleaning StudentLife Sleep...")

# Create a copy of raw dataset
sleep_clean = sleep_raw.copy()

# Remove irrelevant null column if present
if 'null' in sleep_clean.columns:
    sleep_clean = sleep_clean.drop(columns=['null'])

# Display initial dataset
print("\nInitial Sleep Dataset:")
print(sleep_clean)

# Convert response time to timestamp
sleep_clean['timestamp'] = pd.to_datetime(sleep_clean['resp_time'], unit='s')

# Show dataset after timestamp conversion
print("\nAfter converting resp_time to timestamp:")
print(sleep_clean)

\ DATASET: SLEEP
Shows students' self-reported sleep duration in hours

Cleaning StudentLife Sleep...

Initial Sleep Dataset:
       resp_time student_id hour                  location rate social
0     1364121437  Sleep_u54  NaN                       NaN  NaN    NaN
1     1364118985  Sleep_u54  NaN                       NaN  NaN    NaN
2     1364121435  Sleep_u54  NaN                       NaN  NaN    NaN
3     1364121434  Sleep_u54  NaN                       NaN  NaN    NaN
4     1364118978  Sleep_u54  NaN                       NaN  NaN    NaN
...          ...        ...  ...                       ...  ...    ...
1639  1369329215  Sleep_u44   12  43.69213441,-72.26712004    1      3
1640  1369416050  Sleep_u44   11  43.69222786,-72.26719913    1      4
1641  1369503822  Sleep_u44   11  43.69476866,-72.27943547    1      4
1642  1369675713  Sleep_u44   10   43.69341383,-72.2733575    1      4
1643  1370114619  Sleep_u44   12  43.69232465,-72.26672118    1      3

[1644 rows x 6 column

In [29]:
# Clean student_id by removing prefix
sleep_clean['student_id'] = sleep_clean['student_id'].str.replace('Sleep_', '')

# Show dataset after cleaning student_id
print("\nAfter cleaning student_id:")
print(sleep_clean)

# Convert hour column to numeric sleep_hours
sleep_clean['sleep_hours'] = pd.to_numeric(sleep_clean['hour'], errors='coerce')

# Show dataset after converting sleep hours
print("\nAfter converting hour to sleep_hours:")
print(sleep_clean)

# Recover missing sleep hours from null column if available
if 'null' in sleep_clean.columns:
    null_as_num = pd.to_numeric(sleep_clean['null'], errors='coerce')
    fill_mask = sleep_clean['sleep_hours'].isna() & null_as_num.notna()
    if fill_mask.any():
        sleep_clean.loc[fill_mask, 'sleep_hours'] = null_as_num.loc[fill_mask]

# Show dataset after recovery step
print("\nAfter recovering sleep_hours from null if applicable:")
print(sleep_clean)

# Keep only relevant columns and remove missing values
sleep_clean = sleep_clean[['student_id', 'timestamp', 'sleep_hours']].dropna()

# Display final cleaned dataset
print("\nFinal cleaned Sleep dataset:")
print(sleep_clean)


After cleaning student_id:
       resp_time student_id hour                  location rate social  \
0     1364121437        u54  NaN                       NaN  NaN    NaN   
1     1364118985        u54  NaN                       NaN  NaN    NaN   
2     1364121435        u54  NaN                       NaN  NaN    NaN   
3     1364121434        u54  NaN                       NaN  NaN    NaN   
4     1364118978        u54  NaN                       NaN  NaN    NaN   
...          ...        ...  ...                       ...  ...    ...   
1639  1369329215        u44   12  43.69213441,-72.26712004    1      3   
1640  1369416050        u44   11  43.69222786,-72.26719913    1      4   
1641  1369503822        u44   11  43.69476866,-72.27943547    1      4   
1642  1369675713        u44   10   43.69341383,-72.2733575    1      4   
1643  1370114619        u44   12  43.69232465,-72.26672118    1      3   

               timestamp  
0    2013-03-24 10:37:17  
1    2013-03-24 09:56:25  
2 